In [6]:
import os
from functools import partial
from timeit import default_timer as timer
from datetime import timedelta
import ast
import math
import copy


import rasterio
from rasterio.mask import mask
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape, Polygon
import pyproj
from shapely.ops import transform
import numpy as np
import pandas as pd
import geopandas as gpd
from rasterio.plot import show
import tqdm
from shapely.geometry import box
from rasterio.errors import RasterioIOError
from rasterio.enums import Resampling


In [7]:
import sys
sys.path.append('../')
from utils import get_tiles_as_gdf, extract_sample, get_window

In [8]:
target_years = list(range(2000, 2020))

In [9]:
forestloss_dir = '../../Vancutsem_yearloss'
forestloss_tiles = get_tiles_as_gdf(forestloss_dir)

In [10]:
#dataset = 'rubber'
#dataset = 'cocoa'
#dataset = 'mapbiomas'
#dataset = 'vancutsem_regrowth'
#dataset = 'laos_shifting_agriculture'
#dataset = 'coffee_maskell'
dataset = 'mapbiomas_indonesia'
#dataset = 'palm_oil'

In [11]:
if dataset == 'rubber':
    rubber_data_path = '/home/jan/Documents/EPFL/project_2/data/input_datasets/rubber/Rubber_10m'
    label_gdf = get_tiles_as_gdf(rubber_data_path)
    dataset_extent = box(*label_gdf.total_bounds)
    class_mapping = {
        1 : 'rubber'
    }
    dataset_prefix = 2000000

    

elif dataset == 'mapbiomas':
    mapbiomas_path = '/home/jan/Documents/EPFL/drivers/data/mapbiomas/brasil_coverage_2022.tif'
    with rasterio.open(mapbiomas_path) as src:
        dataset_extent = box(*src.bounds)
    label_gdf = gpd.GeoDataFrame({'geometry' : [dataset_extent], 'filename': [mapbiomas_path]})
    
    class_mapping = {
        #1 : 'forest', # superclass
        #3 : 'forest',  # "Forest Formation"
        #10 : 'non-forest natural formation', #incl. wetland, grassland etc. - I think this is the 'natural regrowth'

        # Commodities & drivers
        9 : 'forest plantation',
        15 : 'pasture', 
        39 : 'soy',
        46 : 'coffee',
        35 : 'palm oil',
        30 : 'mining',
        24 : 'urban',
    }
    
    dataset_prefix = 1000000
    

elif dataset == 'cocoa':
    cocoa_file_path = '/home/jan/Documents/EPFL/project_2/data/input_datasets/cocoa_map_th_090.tif'
    with rasterio.open(cocoa_file_path) as src:
        dataset_extent = box(*src.bounds)
    label_gdf = gpd.GeoDataFrame({'geometry' : [dataset_extent], 'filename': [cocoa_file_path]})
    
    class_mapping = {
        1 : 'cocoa'
    }
    dataset_prefix = 3000000


elif dataset == 'vancutsem_regrowth':
    regrowth_data_path = '../data/input_datasets/vancutsem_regrowth/'
    label_gdf = get_tiles_as_gdf(regrowth_data_path)
    dataset_extent = box(*label_gdf.total_bounds)
    class_mapping = {
        30 : 'forest regrowth'
    }
    dataset_prefix = 4000000
    
    
    
elif dataset == 'laos_shifting_agriculture':
    dataset_prefix = 5000000
    
    laos_shifting_ag_data_path = '../data/input_datasets/laos_dis_2020.tif'
    with rasterio.open(laos_shifting_ag_data_path) as src:
        dataset_extent = box(*src.bounds)
    label_gdf = gpd.GeoDataFrame({'geometry' : [dataset_extent], 'filename': [laos_shifting_ag_data_path]})
    
    class_mapping = {
        8 : 'shifting agriculture'
    }
    

elif dataset == 'coffee_maskell':
    dataset_prefix = 7000000
    
    vietnam_coffee_data_path = '../data/input_datasets/coffee_MasketEtAl/classified_binary_2020Oct.tif'
    with rasterio.open(vietnam_coffee_data_path) as src:
        dataset_extent = box(*src.bounds)
    label_gdf = gpd.GeoDataFrame({'geometry' : [dataset_extent], 'filename': [vietnam_coffee_data_path]})
    
    class_mapping = {
        1 : 'coffee'
    }

elif dataset == 'mapbiomas_indonesia':
    dataset_prefix = 8000000
    
    mapb_indo = '../data/input_datasets/mapbiomas_indonesia/indonesia_coverage_2022.tif'
    with rasterio.open(mapb_indo) as src:
        dataset_extent = box(*src.bounds)
    label_gdf = gpd.GeoDataFrame({'geometry' : [dataset_extent], 'filename': [mapb_indo]})
    
    class_mapping = {
        9 : 'forest plantation',
        35 : 'palm oil',
        30 : 'mining',
        26 : 'other', #water
        40 : 'other', #rice
        21 : 'other' #other agriculture
    }    


    
elif dataset == 'palm_oil':
    dataset_prefix = 9000000
    palm_oil_folder = '../data/input_datasets/oil_plantation/High_resolution_global_industrial_and_smallholder_oil_palm_map_for_2019/oil_palm_map'
    
    label_gdf = get_tiles_as_gdf(palm_oil_folder)
    dataset_extent = box(*label_gdf.total_bounds)
    class_mapping = {
        1 : 'palm oil',
        2 : 'palm oil'
    }
    


In [12]:
def write_cocoa_map_with_threshold(_cocoa_file_path, _cocoa_file_path_65, _threshold, _output_path):
    '''
    cocoa_file_path_65 = '/home/jan/Documents/EPFL/project_2/data/input_datasets/cocoa_map_th_065.tif'
    cocoa_file_path = '/home/jan/Documents/EPFL/project_2/data/input_datasets/cocoa_map.tif'
    out_path = cocoa_file_path_65.replace('065', '090')
    '''
    
    with rasterio.open(_cocoa_file_path) as src:
        data = src.read()   

        data[np.where((data > 0) & (data < _threshold))] = 0.

        data[data >= _threshold] = 1.

        data[data < 0] = np.nan
    
    with rasterio.open(_cocoa_file_path_65) as src:
        profile = src.profile
        
    with rasterio.open(_output_path, "w", **profile) as sink:
        sink.write(data)


In [13]:
target_values = [key for key in class_mapping.keys()]


In [14]:
filtered_forestloss = forestloss_tiles.loc[forestloss_tiles.intersects(dataset_extent)]

In [15]:
def write_numpy_array_to_tif(array, geometry, profile, patch_size, output_path):
    transform = rasterio.transform.from_bounds(
        geometry.bounds[0], 
        geometry.bounds[1], 
        geometry.bounds[2], 
        geometry.bounds[3], 
        patch_size[0], 
        patch_size[1])
    
    out_profile = copy.deepcopy(profile)

    out_profile.update({'transform': transform,
                    'width' : patch_size[0],
                    'height' : patch_size[1]
                   })
    
    with rasterio.open(output_path, "w", **out_profile) as sink:
        sink.write(array, 1)

    

In [16]:
filtered_forestloss

,geometry,filename
36,"POLYGON ((100.00000 0.00000, 100.00000 10.0000...",../../Vancutsem_yearloss/tile_N10_E90.tif
37,"POLYGON ((110.00000 0.00000, 110.00000 10.0000...",../../Vancutsem_yearloss/tile_N10_E100.tif
38,"POLYGON ((120.00000 0.00000, 120.00000 10.0000...",../../Vancutsem_yearloss/tile_N10_E110.tif
39,"POLYGON ((130.00000 0.00000, 130.00000 10.0000...",../../Vancutsem_yearloss/tile_N10_E120.tif
51,"POLYGON ((100.00000 -10.00000, 100.00000 0.000...",../../Vancutsem_yearloss/tile_N0_E90.tif
52,"POLYGON ((110.00000 -10.00000, 110.00000 0.000...",../../Vancutsem_yearloss/tile_N0_E100.tif
53,"POLYGON ((120.00000 -10.00000, 120.00000 0.000...",../../Vancutsem_yearloss/tile_N0_E110.tif
54,"POLYGON ((130.00000 -10.00000, 130.00000 0.000...",../../Vancutsem_yearloss/tile_N0_E120.tif
55,"POLYGON ((140.00000 -10.00000, 140.00000 0.000...",../../Vancutsem_yearloss/tile_N0_E130.tif
56,"POLYGON ((150.00000 -10.00000, 150.00000 0.000...",../../Vancutsem_yearloss/tile_N0_E140.tif


In [17]:
patch_rows, patch_cols = 34, 34 #as close as possible to 1km square
min_px_forest_loss = 20 #Pixel size is 30x30m;
min_ratio_for_driver = 0.5 # E.g., for patch with 100 pixels of forest loss, at least min_ratio_for_driver*100 must 


geoms = []
forest_loss_years = []
drivers = []
sampleids = [] 
    
    
sampleid = dataset_prefix


for idx1, label_tile in label_gdf.iterrows(): #Can be a single one (cocoa, mapb) or multiple (rubber)
    print(idx1)
    forestloss_intersection = filtered_forestloss.loc[filtered_forestloss.intersects(label_tile.geometry)]
    
    for idx2, forest_loss_tile in forestloss_intersection.iterrows():
        print(idx2)
        intersection = label_tile.geometry.intersection(forest_loss_tile.geometry)
        label_window = get_window(intersection, label_tile.filename)
        forest_loss_window = get_window(intersection, forest_loss_tile.filename)

        
        with rasterio.open(forest_loss_tile.filename) as src:
            forestloss_data = src.read(1, window=forest_loss_window)
            forestloss_profile = src.profile
        
        with rasterio.open(label_tile.filename) as src:
            label_data = src.read(1,           
                out_shape=(
                    forestloss_data.shape[0],
                    forestloss_data.shape[1]  
                ),
            resampling=Resampling.bilinear,
                window=label_window)

            
        assert forestloss_data.shape == label_data.shape
        
        for i in range(0, forestloss_data.shape[0], patch_rows):
            for j in range(0, forestloss_data.shape[1], patch_cols):
                

                sample = extract_sample(
                            forestloss_data,
                            forestloss_profile,
                            label_data,
                            i,
                            j,
                            patch_rows,
                            patch_cols,
                            target_years,
                            target_values,
                            min_ratio_for_driver,
                            forest_loss_window,
                            min_px_forest_loss)
                
                
                if sample: #If conditions are not met, it returns None; if not, it contains the three items
                    driver_class, forestloss, patch_geometry = sample
                    drivers.append(int(driver_class))

                    geoms.append(patch_geometry)
                    sampleids.append(sampleid)
                    sampleid += 1

        if len(label_gdf) == 1: #For e.g., MapBiomas, i want prints per TMF tile 
            print(f"Items: {len(geoms)}")
    
    print(f"Items: {len(geoms)}")

                

0
36
Items: 38322
37
Items: 101729
38
Items: 131234
39
Items: 133308
51


KeyboardInterrupt: 

In [16]:
gdf = gpd.GeoDataFrame(geometry=geoms)

In [17]:
gdf['driver'] = [class_mapping[val] for val in drivers]
gdf['sampleid'] = sampleids

In [18]:
np.unique(gdf.driver, return_counts=True)

(array(['palm oil'], dtype=object), array([175186]))

In [123]:
#forest_loss_years_list = [ (item_tuple[0].tolist(),item_tuple[1].tolist()) for item_tuple in forest_loss_years]
#gdf['forest_loss_years'] = forest_loss_years_list


In [19]:
gdf['geometry'] = gdf['geometry'].centroid

In [20]:
df = pd.DataFrame(gdf)

df.to_csv(f'../data/labels/{dataset}.csv', index=False)


In [21]:
gdf.to_file(f'../data/tmp/{dataset}.gpkg')

In [22]:
dataset

'palm_oil'

In [46]:
df

,geometry,driver,sampleid
0,POINT (-7.13776 8.19951),cocoa,3000000
1,POINT (-7.83413 7.85132),cocoa,3000001
2,POINT (-8.01739 7.76885),cocoa,3000002
3,POINT (-8.00823 7.76885),cocoa,3000003
4,POINT (-7.98074 7.76885),cocoa,3000004
...,...,...,...
9628,POINT (0.55435 7.79634),cocoa,3009628
9629,POINT (0.57268 7.79634),cocoa,3009629
9630,POINT (0.53602 7.75053),cocoa,3009630
9631,POINT (0.47189 7.31071),cocoa,3009631


In [94]:
#gdf[['geometry', 'main_driver']].to_file('../../data/tmp/cocoa_2000_2020.gpkg')

/tmp/ipykernel_99284/3068935073.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf[['geometry', 'main_driver']].to_file('../../data/tmp/annotation_vancutsem_2000_2020.shp')


In [51]:
def convert_string_to_lists(input_string):
    try:
        lists = ast.literal_eval(input_string)
        if isinstance(lists, tuple) and len(lists) == 2 and all(isinstance(lst, list) for lst in lists):
            return lists
        else:
            return None
    except (SyntaxError, ValueError):
        return None

In [52]:
import re
import ast
def modify_string_multiple_items(input_string):
    # Adjusting regex to match and replace patterns for arrays with multiple items
    modified_string = re.sub(r'array\(\[([0-9, ]+)\](, dtype=\w+\d*)?\)', r'[\1]', input_string)
    # Removing any trailing unmatched parentheses
    modified_string = re.sub(r'\)+$', '', modified_string)
    modified_string = modified_string.replace('(','')
    return ast.literal_eval(modified_string)
    #return [eval(item.strip()) for item in  modified_string.split(',')]

# Testing with the new input
output_str_multiple = modify_string_multiple_items('(array([2017, 2020, 2021], dtype=uint16), array([2, 3, 5]))')
output_str_multiple

([2017, 2020, 2021], [2, 3, 5])

In [53]:
gdf['forest_loss_years'] = gdf['forest_loss_years'].apply(convert_string_to_lists)


### Extract a small subset of the data to get started



In [84]:
df = pd.read_csv(f'../data/labels/{dataset}.csv')

In [85]:
    
def flatten(nested_list):
    return [x for xs in nested_list for x in xs]

if dataset == 'mapbiomas':
    driver_classes = np.unique(df.driver, return_counts=True)[0]
    max_count_per_class = 1000


    indices = []
    for _class in driver_classes:
        try:
            class_samples = df.loc[df.driver == _class].sample(max_count_per_class, random_state=1).index
        except ValueError:
            class_samples = df.loc[df.driver == _class].index
        indices.append(class_samples.tolist())

    indices = flatten(indices)

    subset = df.loc[indices]
else:
    subset = df.sample(1000, random_state=1)

In [90]:
pd.DataFrame(subset).to_csv(f'../data/labels/dataset_small/{dataset}_1000.csv', index=False)
#subset[['geometry', 'main_driver_name']].to_file('../../data/starting_dataset/cocoa_1000.gpkg')

In [ ]:
if data